# SpendLens v2 — Procurement Intelligence Dashboard

**What changed in v2:**
- Navy/white enterprise theme (replaces dark mode)
- Revised spend taxonomy (SaaS company — no manufacturing)
- Capex vs Opex classification
- Year selector — filters all charts
- Supplier drill-down — click any bubble to see top suppliers behind it
- Spend by region chart
- Fixed label overlaps on stacked area and gauges
- New KPIs: PO coverage, contract coverage, supplier count

**Replace instructions at the bottom of this notebook (Section 10).**


## 0. Setup
Run once. Skip if packages already installed.

In [ ]:
!pip install panel holoviews hvplot plotly pandas openpyxl bokeh param anthropic --quiet
print("✅ Done")

: 

## 1. Imports & Configuration

In [ ]:
import panel as pn
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import param, io, os, re, json
from datetime import datetime

pn.extension("plotly", sizing_mode="stretch_width")
print(f"✅ Panel {pn.__version__} | Pandas {pd.__version__}")

## 2. Theme — Navy / White Enterprise

Clean professional theme inspired by government/finance dashboards.
Deep navy accent, white surfaces, minimal borders.


In [ ]:
# ── Core palette ──
BG       = "#FFFFFF"    # Page background — pure white
CARD     = "#F8F9FA"    # Card/chart background — very light grey
GRID     = "#E9ECEF"    # Grid lines and borders
TEXT     = "#1A1A2E"    # Primary text — near black
DIM      = "#6C757D"    # Secondary/muted text
NAVY     = "#1B3A6B"    # Primary accent — deep navy
NAVY2    = "#2E5BA8"    # Secondary accent — medium blue
GREEN    = "#1A7A4A"    # Good / Low risk
YELLOW   = "#B8860B"    # Warning / Medium risk (dark gold, readable on white)
RED      = "#C0392B"    # Danger / Critical risk
ORANGE   = "#D35400"    # Medium-high risk

# 10-color palette for categories (all readable on white)
COLORS = ["#1B3A6B", "#2E5BA8", "#1A7A4A", "#B8860B", "#6A3EA1",
          "#0E7C86", "#C0392B", "#D35400", "#2C7873", "#4A235A"]

RISK_COLORS = {
    "Critical": RED,
    "High":     YELLOW,
    "Medium":   ORANGE,
    "Low":      GREEN,
}

# ── Reusable Plotly layout — applied to every chart ──
LAYOUT = dict(
    paper_bgcolor=BG,
    plot_bgcolor=CARD,
    font=dict(family="Arial, sans-serif", color=TEXT, size=12),
    margin=dict(l=60, r=30, t=50, b=40),
    xaxis=dict(gridcolor=GRID, linecolor=GRID, tickfont=dict(size=11)),
    yaxis=dict(gridcolor=GRID, linecolor=GRID, tickfont=dict(size=11)),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=10), orientation="v"),
)

# ── CSS injected into Panel app ──
CSS = """
body { background-color: #FFFFFF !important; color: #1A1A2E !important; }
.bk-root { background-color: #FFFFFF !important; }
.fast-header { background-color: #1B3A6B !important; }
"""
pn.config.raw_css.append(CSS)

print("🎨 Theme configured — Navy/White Enterprise")

## 3. Data Model — Revised SaaS Taxonomy

10 categories relevant to a SaaS/AI company.
Each category has:
- Spend per year (2022-2026E)
- Risk metadata (concentration, lead time, single source)
- Capex/Opex classification
- Region (budget owner location)
- Supplier breakdown (realistic splits for drill-down)


In [ ]:
YEARS     = [2022, 2023, 2024, 2025, 2026]
HEADCOUNT = [49,   120,  220,  380,  600]

# ── Category definitions ──
# spend: €K per year | concentration: % held by top supplier
# capex_opex: "Capex" or "Opex" | region: budget owner
categories_raw = [
    {
        "name": "Cloud & Compute",
        "spend": [4200, 7800, 12500, 17800, 24000],
        "concentration": 72, "risk": "Critical", "single_source": False,
        "lead_time_days": 0, "contract_end": "2026-09",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 3,
        # Supplier breakdown: {name, spend_pct, region, contract_end}
        "suppliers": [
            {"name": "Amazon Web Services", "pct": 60, "contract_end": "2026-09"},
            {"name": "Google Cloud Platform", "pct": 25, "contract_end": "2027-03"},
            {"name": "Microsoft Azure",       "pct": 15, "contract_end": "2026-12"},
        ]
    },
    {
        "name": "AI/ML APIs & Data",
        "spend": [800, 2200, 4800, 6500, 9200],
        "concentration": 55, "risk": "High", "single_source": False,
        "lead_time_days": 14, "contract_end": "2026-06",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 5,
        "suppliers": [
            {"name": "OpenAI",        "pct": 40, "contract_end": "2026-06"},
            {"name": "Anthropic",     "pct": 25, "contract_end": "2026-12"},
            {"name": "Scale AI",      "pct": 20, "contract_end": "2027-01"},
            {"name": "Hugging Face",  "pct": 10, "contract_end": "2026-09"},
            {"name": "Cohere",        "pct":  5, "contract_end": "2026-08"},
        ]
    },
    {
        "name": "IT Software & SaaS",
        "spend": [900, 1400, 2200, 3100, 4200],
        "concentration": 22, "risk": "Low", "single_source": False,
        "lead_time_days": 7, "contract_end": "Various",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 18,
        "suppliers": [
            {"name": "GitHub / Copilot",  "pct": 22, "contract_end": "2026-11"},
            {"name": "Datadog",           "pct": 18, "contract_end": "2027-01"},
            {"name": "Atlassian",         "pct": 15, "contract_end": "2026-10"},
            {"name": "Slack / Salesforce","pct": 12, "contract_end": "2026-12"},
            {"name": "Notion",            "pct":  8, "contract_end": "2026-09"},
            {"name": "Other SaaS tools",  "pct": 25, "contract_end": "Various"},
        ]
    },
    {
        "name": "Telecom & Voice Infra",
        "spend": [400, 800, 1400, 2200, 3000],
        "concentration": 65, "risk": "Critical", "single_source": True,
        "lead_time_days": 30, "contract_end": "2026-07",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 3,
        "suppliers": [
            {"name": "Twilio",            "pct": 65, "contract_end": "2026-07"},
            {"name": "Vonage / Ericsson", "pct": 25, "contract_end": "2027-02"},
            {"name": "Deutsche Telekom",  "pct": 10, "contract_end": "2027-12"},
        ]
    },
    {
        "name": "Recruitment & HR",
        "spend": [600, 1100, 2400, 4200, 6800],
        "concentration": 35, "risk": "High", "single_source": False,
        "lead_time_days": 45, "contract_end": "2026-12",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 8,
        "suppliers": [
            {"name": "Personio (HRIS)",     "pct": 20, "contract_end": "2027-03"},
            {"name": "LinkedIn Talent",     "pct": 35, "contract_end": "2026-12"},
            {"name": "Kienbaum / Kearney",  "pct": 15, "contract_end": "2026-09"},
            {"name": "Staffing agencies",   "pct": 20, "contract_end": "Various"},
            {"name": "Other job boards",    "pct": 10, "contract_end": "Various"},
        ]
    },
    {
        "name": "Professional Services",
        "spend": [400, 700, 1200, 2100, 3200],
        "concentration": 40, "risk": "Medium", "single_source": False,
        "lead_time_days": 21, "contract_end": "2026-03",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 7,
        "suppliers": [
            {"name": "KPMG",                    "pct": 30, "contract_end": "2026-06"},
            {"name": "Deloitte",                "pct": 25, "contract_end": "2026-03"},
            {"name": "EY",                      "pct": 15, "contract_end": "2026-09"},
            {"name": "Baker McKenzie (Legal)",  "pct": 20, "contract_end": "2026-12"},
            {"name": "Impl. partners",          "pct": 10, "contract_end": "Various"},
        ]
    },
    {
        "name": "Marketing & Campaigns",
        "spend": [300, 800, 1800, 3500, 5500],
        "concentration": 28, "risk": "Medium", "single_source": False,
        "lead_time_days": 30, "contract_end": "Various",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 10,
        "suppliers": [
            {"name": "Google Ads",        "pct": 28, "contract_end": "Rolling"},
            {"name": "LinkedIn Ads",      "pct": 22, "contract_end": "Rolling"},
            {"name": "Event agencies",    "pct": 20, "contract_end": "Various"},
            {"name": "PR agency",         "pct": 15, "contract_end": "2026-06"},
            {"name": "Content / Design",  "pct": 15, "contract_end": "Various"},
        ]
    },
    {
        "name": "Facilities & Office",
        "spend": [500, 900, 1500, 2800, 4800],
        "concentration": 55, "risk": "High", "single_source": False,
        "lead_time_days": 90, "contract_end": "2028-06",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 4,
        "suppliers": [
            {"name": "Berlin HQ landlord",   "pct": 55, "contract_end": "2028-06"},
            {"name": "WeWork / Flex space",  "pct": 20, "contract_end": "2026-12"},
            {"name": "Cleaning & security",  "pct": 15, "contract_end": "2026-09"},
            {"name": "Office supplies",      "pct": 10, "contract_end": "Various"},
        ]
    },
    {
        "name": "Hardware & Equipment",
        "spend": [300, 500, 900, 1500, 2400],
        "concentration": 45, "risk": "Medium", "single_source": False,
        "lead_time_days": 56, "contract_end": "N/A",
        "capex_opex": "Capex", "region": "DACH",
        "supplier_count": 4,
        "suppliers": [
            {"name": "Apple (MacBooks)",    "pct": 45, "contract_end": "N/A"},
            {"name": "Lenovo",              "pct": 25, "contract_end": "N/A"},
            {"name": "Peripherals / Audio", "pct": 20, "contract_end": "N/A"},
            {"name": "Monitors / Docking",  "pct": 10, "contract_end": "N/A"},
        ]
    },
    {
        "name": "Travel & Expenses",
        "spend": [200, 500, 1100, 2000, 3200],
        "concentration": 30, "risk": "Low", "single_source": False,
        "lead_time_days": 3, "contract_end": "2026-09",
        "capex_opex": "Opex", "region": "DACH",
        "supplier_count": 6,
        "suppliers": [
            {"name": "Navan (TMC platform)", "pct": 30, "contract_end": "2026-09"},
            {"name": "Lufthansa / Airlines", "pct": 25, "contract_end": "Rolling"},
            {"name": "Hotels",               "pct": 20, "contract_end": "Rolling"},
            {"name": "Corporate cards",      "pct": 15, "contract_end": "2027-01"},
            {"name": "Rail / local transit", "pct": 10, "contract_end": "Rolling"},
        ]
    },
]

# ── Build DataFrames ──

# Spend over time: one row per (category, year)
spend_rows = []
for cat in categories_raw:
    for i, year in enumerate(YEARS):
        spend_rows.append({
            "category":    cat["name"],
            "year":        year,
            "spend":       cat["spend"][i],
            "capex_opex":  cat["capex_opex"],
            "region":      cat["region"],
        })
df_spend = pd.DataFrame(spend_rows)

# Category metadata: one row per category
meta_rows = [{
    "category":       c["name"],
    "spend_2026e":    c["spend"][4],
    "concentration":  c["concentration"],
    "risk":           c["risk"],
    "single_source":  c["single_source"],
    "lead_time_days": c["lead_time_days"],
    "contract_end":   c["contract_end"],
    "capex_opex":     c["capex_opex"],
    "region":         c["region"],
    "supplier_count": c["supplier_count"],
    "cagr":           round(((c["spend"][4] / c["spend"][0]) ** (1/4) - 1) * 100, 1),
    "suppliers":      c["suppliers"],   # kept for drill-down
} for c in categories_raw]
df_meta = pd.DataFrame(meta_rows)

# Headcount & totals
df_headcount = pd.DataFrame({"year": YEARS, "headcount": HEADCOUNT})
df_totals = df_spend.groupby("year")["spend"].sum().reset_index()
df_totals.columns = ["year", "total_spend"]
df_totals = df_totals.merge(df_headcount, on="year")
df_totals["spend_per_head"] = (df_totals["total_spend"] / df_totals["headcount"]).round(1)

# Capex vs Opex summary
df_capex = df_spend.groupby(["year", "capex_opex"])["spend"].sum().reset_index()

print(f"✅ Data loaded — {len(df_meta)} categories")
print(f"   Total 2026E spend: €{df_meta['spend_2026e'].sum():,.0f}K")
print(f"   Capex: €{df_meta[df_meta.capex_opex=='Capex']['spend_2026e'].sum():,.0f}K")
print(f"   Opex:  €{df_meta[df_meta.capex_opex=='Opex']['spend_2026e'].sum():,.0f}K")
print(f"   Total suppliers: {df_meta['supplier_count'].sum()}")

## 4. Chart Functions

All charts are reusable functions. The year selector widget calls these with filtered data.
Bug fixes applied: label overlap on stacked area, gauge title/number spacing.


In [ ]:
def chart_spend_vs_headcount(df_totals):
    """Dual-axis: spend bars + headcount line."""
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(
        x=df_totals["year"], y=df_totals["total_spend"],
        name="Total Spend (€K)", marker_color=NAVY, opacity=0.85,
        # FIX: show labels only outside, texttemplate avoids overlap
        text=[f"€{v/1000:.1f}M" for v in df_totals["total_spend"]],
        textposition="outside", textfont=dict(size=10),
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=df_totals["year"], y=df_totals["headcount"],
        name="Headcount", mode="lines+markers",
        line=dict(color=RED, width=2.5, dash="dot"), marker=dict(size=7),
    ), secondary_y=True)
    fig.update_layout(title="Total Spend vs Headcount", **LAYOUT, height=380,
                      legend=dict(orientation="h", y=-0.15))
    fig.update_yaxes(title_text="Spend (€K)", secondary_y=False, gridcolor=GRID)
    fig.update_yaxes(title_text="Headcount", secondary_y=True, showgrid=False)
    return fig


def chart_spend_stacked(df_spend_filtered, year=None):
    """
    Stacked area chart by category.
    FIX: removed textlabels that caused overlap — use hover instead.
    """
    fig = go.Figure()
    cats = df_spend_filtered.groupby("category")["spend"].sum().sort_values().index
    for i, cat in enumerate(cats):
        d = df_spend_filtered[df_spend_filtered["category"] == cat]
        fig.add_trace(go.Scatter(
            x=d["year"], y=d["spend"], name=cat[:22], mode="lines",
            stackgroup="one",
            line=dict(width=0.5, color=COLORS[i % len(COLORS)]),
            hovertemplate=f"<b>{cat}</b><br>Year: %{{x}}<br>Spend: €%{{y:,.0f}}K<extra></extra>",
        ))
    title = f"Spend by Category — {year}" if year else "Spend Evolution by Category"
    fig.update_layout(title=title, **LAYOUT, height=420, yaxis_title="Spend (€K)",
                      legend=dict(orientation="v", x=1.01, y=1))
    return fig


def chart_category_bars(df_meta_filtered):
    """Horizontal bar — spend per category ranked."""
    df = df_meta_filtered.sort_values("spend_2026e", ascending=True)
    fig = go.Figure(go.Bar(
        y=df["category"], x=df["spend_2026e"], orientation="h",
        marker_color=[COLORS[i % len(COLORS)] for i in range(len(df))],
        text=[f"€{v:,.0f}K" for v in df["spend_2026e"]],
        textposition="outside", textfont=dict(size=10),
    ))
    fig.update_layout(title="Spend by Category", **LAYOUT, height=400,
                      xaxis_title="Spend (€K)", margin=dict(l=160, r=80, t=50, b=40))
    return fig


def chart_cagr(df_meta_filtered):
    """CAGR per category — growth rate 2022→2026E."""
    df = df_meta_filtered.sort_values("cagr", ascending=True)
    colors = [RED if v > 40 else YELLOW if v > 25 else NAVY for v in df["cagr"]]
    fig = go.Figure(go.Bar(
        y=df["category"], x=df["cagr"], orientation="h",
        marker_color=colors,
        text=[f"{v:.1f}%" for v in df["cagr"]], textposition="outside",
    ))
    fig.update_layout(title="Category CAGR (2022→2026E)", **LAYOUT, height=400,
                      xaxis_title="CAGR %", margin=dict(l=160, r=60, t=50, b=40))
    return fig


def chart_risk_bubble(df_meta_filtered):
    """
    Bottleneck bubble map.
    X = concentration, Y = spend, bubble = lead time, color = risk.
    CLICK on a bubble triggers the drill-down panel.
    """
    fig = go.Figure()
    for _, row in df_meta_filtered.iterrows():
        fig.add_trace(go.Scatter(
            x=[row["concentration"]], y=[row["spend_2026e"]],
            mode="markers+text",
            marker=dict(
                size=max(row["lead_time_days"] + 18, 20),
                color=RISK_COLORS.get(row["risk"], DIM),
                opacity=0.75,
                line=dict(width=1.5, color="white"),
            ),
            text=[row["category"][:16]], textposition="top center",
            textfont=dict(size=9, color=TEXT),
            name=row["category"],
            customdata=[[row["category"], row["risk"],
                         row["supplier_count"], row["contract_end"]]],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Concentration: %{x}%<br>"
                "Spend: €%{y:,.0f}K<br>"
                "Lead time: " + str(row["lead_time_days"]) + "d<br>"
                "Suppliers: " + str(row["supplier_count"]) + "<br>"
                "Contract end: " + str(row["contract_end"]) + "<br>"
                "Risk: %{customdata[1]}<extra></extra>"
            ),
        ))

    # Danger zone
    fig.add_shape(type="rect", x0=50, x1=100, y0=0, y1=30000,
                  fillcolor="rgba(192,57,43,0.05)",
                  line=dict(color="rgba(192,57,43,0.4)", dash="dot", width=1.5))
    fig.add_annotation(x=75, y=27000, text="⚠ Danger Zone",
                       showarrow=False, font=dict(size=11, color=RED))

    fig.update_layout(
        title="Bottleneck Map — Click a bubble to see supplier detail",
        **LAYOUT, height=520, showlegend=False,
        xaxis=dict(title="Supplier Concentration (%)", range=[0, 105],
                   gridcolor=GRID, linecolor=GRID),
        yaxis=dict(title="Spend (€K)", gridcolor=GRID, linecolor=GRID),
    )
    return fig


def chart_treemap(df_meta_filtered):
    """Treemap: size = spend, color = concentration."""
    df = df_meta_filtered.copy()
    df["label"] = df.apply(
        lambda r: f"{r['category']}<br>€{r['spend_2026e']:,.0f}K", axis=1)
    fig = px.treemap(df, path=["label"], values="spend_2026e", color="concentration",
                     color_continuous_scale=["#1A7A4A", "#B8860B", "#C0392B"],
                     range_color=[0, 80])
    fig.update_layout(title="Spend (size) × Concentration (color)",
                      paper_bgcolor=BG, font=dict(color=TEXT, size=12), height=420,
                      coloraxis_colorbar=dict(title="Conc %", tickfont=dict(size=10)))
    return fig


def chart_capex_opex(df_spend_filtered):
    """Grouped bar: Capex vs Opex split per year."""
    df = df_spend_filtered.groupby(["year", "capex_opex"])["spend"].sum().reset_index()
    fig = go.Figure()
    for co, color in [("Opex", NAVY), ("Capex", NAVY2)]:
        d = df[df["capex_opex"] == co]
        fig.add_trace(go.Bar(
            x=d["year"], y=d["spend"], name=co,
            marker_color=color, opacity=0.85,
            text=[f"€{v:,.0f}K" for v in d["spend"]],
            textposition="outside", textfont=dict(size=9),
        ))
    fig.update_layout(title="Capex vs Opex Split", **LAYOUT,
                      height=350, barmode="group", yaxis_title="Spend (€K)")
    return fig


def chart_region(df_spend_filtered):
    """Spend by region (budget owner) — bar chart."""
    df = df_spend_filtered.groupby("region")["spend"].sum().reset_index()
    df = df.sort_values("spend", ascending=True)
    fig = go.Figure(go.Bar(
        y=df["region"], x=df["spend"], orientation="h",
        marker_color=NAVY,
        text=[f"€{v:,.0f}K" for v in df["spend"]], textposition="outside",
    ))
    fig.update_layout(title="Spend by Region (budget owner)", **LAYOUT,
                      height=250, xaxis_title="Spend (€K)")
    return fig


def chart_gauges(df_meta_filtered):
    """
    Risk scorecard gauges.
    FIX: title font reduced, number font size adjusted to prevent overlap.
    """
    avg_conc  = df_meta_filtered["concentration"].mean()
    criticals = len(df_meta_filtered[df_meta_filtered["risk"] == "Critical"])
    singles   = int(df_meta_filtered["single_source"].sum())
    avg_lead  = df_meta_filtered["lead_time_days"].mean()
    overall   = round(avg_conc * 0.4 + criticals * 12 + singles * 8 + avg_lead * 0.25)

    fig = make_subplots(
        rows=1, cols=5,
        specs=[[{"type": "indicator"}] * 5],
        # FIX: use column_titles instead of subplot_titles to avoid overlap
        column_titles=["Overall Risk", "Avg Concentration", "Critical", "Single-Source", "Avg Lead Time"],
        horizontal_spacing=0.05,
    )
    gauges = [(overall, 100), (avg_conc, 100), (criticals, 5), (singles, 5), (avg_lead, 90)]
    for i, (val, ref) in enumerate(gauges):
        ratio = val / ref
        color = RED if ratio > 0.6 else YELLOW if ratio > 0.35 else GREEN
        fig.add_trace(go.Indicator(
            mode="gauge+number",
            value=round(val, 1),
            # FIX: smaller number font prevents overlap with title
            number=dict(font=dict(size=22, color=color)),
            gauge=dict(
                axis=dict(range=[0, ref], tickfont=dict(size=9)),
                bar=dict(color=color, thickness=0.6),
                bgcolor=CARD,
                bordercolor=GRID,
                steps=[
                    dict(range=[0, ref * 0.35], color="#EAF3DE"),
                    dict(range=[ref * 0.35, ref * 0.6], color="#FDF3DC"),
                    dict(range=[ref * 0.6, ref], color="#FCEAE8"),
                ],
            ),
        ), row=1, col=i + 1)

    fig.update_layout(
        paper_bgcolor=BG,
        font=dict(color=TEXT, size=10),
        height=220,
        margin=dict(l=10, r=10, t=60, b=10),
    )
    return fig


print("✅ All chart functions defined")

## 5. KPI Cards

In [ ]:
def kpi_card(title, value, subtitle="", color=NAVY, bg=CARD):
    """Styled KPI metric card — navy/white theme."""
    return pn.pane.HTML(f"""
    <div style="background:{bg}; border-radius:8px; padding:16px 20px;
                border:1px solid {GRID}; min-width:140px; text-align:center;">
        <div style="color:{DIM}; font-size:11px; text-transform:uppercase;
                    letter-spacing:0.08em; margin-bottom:6px;">{title}</div>
        <div style="color:{color}; font-size:28px; font-weight:600;
                    margin:4px 0; line-height:1.1;">{value}</div>
        <div style="color:{DIM}; font-size:11px; margin-top:4px;">{subtitle}</div>
    </div>
    """, sizing_mode="stretch_width")

print("✅ KPI cards ready")

## 6. Supplier Drill-Down Panel

When a bubble is clicked on the bottleneck map, this panel renders
the top suppliers for that category with spend amounts and contract dates.


In [ ]:
def render_drill_down(category_name, df_meta):
    """
    Render a supplier detail panel for the clicked category.
    Shows: supplier name, spend %, estimated €K, contract end.
    """
    row = df_meta[df_meta["category"] == category_name]
    if row.empty:
        return pn.pane.Markdown("No data found.")

    row = row.iloc[0]
    suppliers = row["suppliers"]
    total_spend = row["spend_2026e"]
    risk_color = RISK_COLORS.get(row["risk"], DIM)

    # Build supplier rows HTML
    rows_html = ""
    for i, s in enumerate(suppliers[:20]):
        est_spend = round(total_spend * s["pct"] / 100)
        bg = CARD if i % 2 == 0 else "#FFFFFF"
        rows_html += f"""
        <tr style="background:{bg};">
            <td style="padding:8px 12px; font-size:13px;">{s['name']}</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;">
                {s['pct']}%</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;
                       color:{NAVY}; font-weight:500;">
                €{est_spend:,.0f}K</td>
            <td style="padding:8px 12px; font-size:12px; color:{DIM};">
                {s['contract_end']}</td>
        </tr>"""

    html = f"""
    <div style="background:{BG}; border:1px solid {GRID}; border-radius:8px;
                padding:16px 20px; margin-top:12px;">
        <div style="display:flex; align-items:center; gap:12px; margin-bottom:12px;">
            <div style="width:10px; height:10px; border-radius:50%;
                        background:{risk_color};"></div>
            <span style="font-size:16px; font-weight:600; color:{NAVY};">
                {category_name}</span>
            <span style="font-size:13px; color:{DIM};">
                — {row['risk']} risk | {row['supplier_count']} suppliers |
                Conc. {row['concentration']}% | Lead time {row['lead_time_days']}d
            </span>
        </div>
        <table style="width:100%; border-collapse:collapse;
                      border:0.5px solid {GRID}; border-radius:6px; overflow:hidden;">
            <thead>
                <tr style="background:{NAVY}; color:white;">
                    <th style="padding:8px 12px; text-align:left;
                               font-size:12px; font-weight:500;">Supplier</th>
                    <th style="padding:8px 12px; text-align:right;
                               font-size:12px; font-weight:500;">Share</th>
                    <th style="padding:8px 12px; text-align:right;
                               font-size:12px; font-weight:500;">Est. Spend</th>
                    <th style="padding:8px 12px; text-align:left;
                               font-size:12px; font-weight:500;">Contract End</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
        <div style="margin-top:8px; font-size:11px; color:{DIM};">
            * Spend estimates based on % split of 2026E category total (€{total_spend:,.0f}K)
        </div>
    </div>
    """
    return pn.pane.HTML(html, sizing_mode="stretch_width")

print("✅ Drill-down panel ready")

## 7. Launch Dashboard

Run this cell to open the full dashboard in your browser.

**New in v2:**
- Year selector in the sidebar — all charts update when you change it
- Click any bubble on the bottleneck map → supplier drill-down appears below
- Capex vs Opex chart
- Spend by region
- Fixed gauges and label overlaps


In [ ]:
# ── Sidebar widgets ──
year_select = pn.widgets.Select(
    name="Year filter",
    options=["All years"] + YEARS,
    value="All years",
    width=250,
)
file_input = pn.widgets.FileInput(accept=".csv,.xlsx,.xls", name="Upload spend data")
dataset_label = pn.pane.Markdown("**Dataset:** Default (SaaS taxonomy)",
                                  styles={"color": TEXT, "font-size": "13px"})
api_key_input = pn.widgets.PasswordInput(
    name="Claude API key (optional)", placeholder="sk-ant-...", width=250)
export_btn = pn.widgets.Button(
    name="📥 Export CFO Report", button_type="primary", width=250)
status_log = pn.pane.Markdown("", styles={"color": DIM, "font-size": "11px"})

# ── Main content areas ──
kpi_row      = pn.Row(sizing_mode="stretch_width")
chart_area   = pn.Column(sizing_mode="stretch_width")
drill_panel  = pn.Column(sizing_mode="stretch_width")   # drill-down lives here
data_preview = pn.widgets.Tabulator(
    df_meta.drop(columns=["suppliers"]),
    sizing_mode="stretch_width", height=280,
    theme="bootstrap", page_size=12,
)


def get_filtered_data(year_val):
    """Return filtered DataFrames based on year selector."""
    if year_val == "All years":
        # For meta, use 2026E as reference year
        dm = df_meta.copy()
        ds = df_spend.copy()
    else:
        # Filter spend to selected year only
        ds = df_spend[df_spend["year"] == year_val].copy()
        # Rebuild meta spend from selected year
        dm = df_meta.copy()
        year_spend = ds.groupby("category")["spend"].sum()
        dm["spend_2026e"] = dm["category"].map(year_spend).fillna(0)
    return dm, ds


def update_dashboard(year_val=None):
    """Refresh all KPIs and charts."""
    if year_val is None:
        year_val = year_select.value

    dm, ds = get_filtered_data(year_val)
    year_label = str(year_val) if year_val != "All years" else "2026E"

    # KPIs
    total     = dm["spend_2026e"].sum()
    crits     = len(dm[dm["risk"] == "Critical"])
    singles   = int(dm["single_source"].sum())
    avg_c     = dm["concentration"].mean()
    sup_total = dm["supplier_count"].sum()
    capex_amt = dm[dm["capex_opex"] == "Capex"]["spend_2026e"].sum()
    opex_pct  = round((1 - capex_amt / total) * 100, 0) if total > 0 else 0

    kpi_row.clear()
    kpi_row.extend([
        kpi_card("Total Spend", f"€{total/1000:.1f}M", year_label),
        kpi_card("Critical Risks", str(crits), "categories",
                 RED if crits > 0 else GREEN),
        kpi_card("Single Source", str(singles), "dependencies",
                 YELLOW if singles > 0 else GREEN),
        kpi_card("Avg Concentration", f"{avg_c:.0f}%", "top supplier share",
                 RED if avg_c > 50 else YELLOW if avg_c > 35 else GREEN),
        kpi_card("Suppliers", str(sup_total), "total across categories"),
        kpi_card("Opex Share", f"{opex_pct:.0f}%", f"Capex €{capex_amt/1000:.1f}M"),
    ])

    # Charts
    chart_area.clear()
    chart_area.extend([
        pn.pane.Plotly(chart_gauges(dm), sizing_mode="stretch_width"),
        pn.pane.Plotly(chart_spend_vs_headcount(df_totals), sizing_mode="stretch_width"),
        pn.pane.Plotly(chart_spend_stacked(ds, year_val if year_val != "All years" else None),
                       sizing_mode="stretch_width"),
        pn.Row(
            pn.pane.Plotly(chart_category_bars(dm), sizing_mode="stretch_width"),
            pn.pane.Plotly(chart_cagr(dm), sizing_mode="stretch_width"),
        ),
        pn.pane.Plotly(chart_risk_bubble(dm), sizing_mode="stretch_width"),
        drill_panel,   # drill-down renders here on click
        pn.Row(
            pn.pane.Plotly(chart_capex_opex(ds), sizing_mode="stretch_width"),
            pn.pane.Plotly(chart_region(ds), sizing_mode="stretch_width"),
        ),
        pn.pane.Plotly(chart_treemap(dm), sizing_mode="stretch_width"),
    ])

    data_preview.value = dm.drop(columns=["suppliers"])


# ── Year selector callback ──
def on_year_change(event):
    update_dashboard(event.new)

year_select.param.watch(on_year_change, "value")


# ── Bubble click drill-down ──
# Panel Plotly pane captures click events via selected_data
bubble_pane = pn.pane.Plotly(
    chart_risk_bubble(df_meta), sizing_mode="stretch_width")

def on_bubble_click(event):
    """When user clicks a bubble, render supplier drill-down below the chart."""
    if event.new and "points" in event.new and event.new["points"]:
        pt = event.new["points"][0]
        cat_name = pt.get("text", "").replace("<br>", " ").strip()
        # Try customdata first, fallback to text matching
        if "customdata" in pt and pt["customdata"]:
            cat_name = pt["customdata"][0]
        drill_panel.clear()
        drill_panel.append(render_drill_down(cat_name, df_meta))

bubble_pane.param.watch(on_bubble_click, "selected_data")


# ── File upload ──
def handle_upload(event):
    if file_input.value is None:
        return
    try:
        raw = io.BytesIO(file_input.value)
        fname = file_input.filename
        df_new = pd.read_csv(raw) if fname.endswith(".csv") else pd.read_excel(raw)
        status_log.object = f"✅ Loaded **{fname}** — {len(df_new)} rows"
        dataset_label.object = f"**Dataset:** {fname}"
        data_preview.value = df_new.head(50)
    except Exception as e:
        status_log.object = f"❌ {str(e)}"

file_input.param.watch(handle_upload, "value")


# ── Export ──
def handle_export(event):
    try:
        buffer = io.BytesIO()
        with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
            df_meta.drop(columns=["suppliers"]).to_excel(
                writer, sheet_name="Category Summary", index=False)
            df_spend.to_excel(writer, sheet_name="Spend Over Time", index=False)
            df_capex.to_excel(writer, sheet_name="Capex vs Opex", index=False)
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        fname = f"CFO_Spend_Report_{ts}.xlsx"
        dl = pn.widgets.FileDownload(
            file=io.BytesIO(buffer.getvalue()),
            filename=fname, button_type="success", label="⬇ Download")
        status_log.object = f"✅ Ready: **{fname}**"
        sidebar_col.append(dl)
    except Exception as e:
        status_log.object = f"❌ {str(e)}"

export_btn.on_click(handle_export)


# ── Initial render ──
update_dashboard()

# ── Header ──
header = pn.pane.HTML(f"""
<div style="background:{NAVY}; padding:20px 28px; border-radius:8px; margin-bottom:16px;">
    <h1 style="color:white; margin:0; font-size:24px; font-weight:600;">
        SpendLens</h1>
    <p style="color:rgba(255,255,255,0.7); margin:4px 0 0 0; font-size:13px;">
        Procurement Intelligence Dashboard — Upload any spend data for instant analysis
    </p>
</div>
""", sizing_mode="stretch_width")

# ── Sidebar ──
sidebar_col = pn.Column(
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>YEAR</div>"),
    year_select,
    pn.layout.Divider(),
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>DATA SOURCE</div>"),
    file_input, dataset_label,
    pn.layout.Divider(),
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>AI SETTINGS</div>"),
    api_key_input,
    pn.layout.Divider(),
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>REPORTS</div>"),
    export_btn,
    pn.layout.Divider(),
    status_log,
    width=280,
)

# ── Main ──
main_col = pn.Column(
    header, kpi_row, pn.layout.Divider(),
    chart_area, pn.layout.Divider(),
    pn.pane.HTML(f"<h3 style='color:{NAVY};'>Data Explorer</h3>"),
    data_preview,
    sizing_mode="stretch_width",
)

# ── Template ──
template = pn.template.FastListTemplate(
    title="SpendLens",
    sidebar=sidebar_col,
    main=[main_col],
    accent_base_color=NAVY,
    header_background=NAVY,
    background_color=BG,
    theme="default",   # light theme
)

template.show()
print("\n🚀 Dashboard open — http://localhost:5006")

## 8. AI Column Mapper
*(unchanged from v1 — maps messy column names to standard schema)*

In [ ]:
STANDARD_SCHEMA = {
    "category":       ["category","spend_category","procurement_category","cost_center",
                       "warengruppe","kategorie","commodity","group"],
    "supplier":       ["supplier","vendor","provider","lieferant","kreditor","creditor",
                       "supplier_name","vendor_name","company","partner"],
    "spend":          ["spend","amount","total","value","cost","betrag","invoice_total",
                       "invoice_amount","rechnungsbetrag","net_amount","spend_eur",
                       "spend_amount","total_spend"],
    "currency":       ["currency","curr","währung","ccy"],
    "date":           ["date","invoice_date","datum","period","month","year",
                       "posting_date","buchungsdatum","doc_date"],
    "contract_end":   ["contract_end","contract_expiry","expiry_date","end_date",
                       "vertragslaufzeit","renewal_date"],
    "risk":           ["risk","risk_level","risiko","risk_rating"],
    "region":         ["region","country","location","land","geography","geo"],
    "department":     ["department","dept","business_unit","bu","abteilung","org_unit"],
    "po_number":      ["po_number","purchase_order","bestellnummer","po","order_number"],
    "capex_opex":     ["capex_opex","capex","opex","asset_type","cost_type"],
}

def rule_based_mapping(columns):
    mapping = {}
    normalized = {c: c.strip().lower().replace(" ","_").replace("-","_") for c in columns}
    for orig, norm in normalized.items():
        matched = False
        for std, syns in STANDARD_SCHEMA.items():
            if norm in syns or any(s in norm for s in syns):
                mapping[orig] = std; matched = True; break
        if not matched:
            mapping[orig] = None
    return mapping

def ai_column_mapping(columns, sample_rows, api_key=None):
    mapping = rule_based_mapping(columns)
    unknowns = [c for c, v in mapping.items() if v is None]
    if not unknowns: return mapping
    api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return {k: v for k, v in mapping.items() if v}
    schema_desc = "\n".join([f"  - {k}: {\', \'.join(v[:4])}" for k, v in STANDARD_SCHEMA.items()])
    prompt = f"""Map these column names to the standard procurement schema.
Schema: {schema_desc}
Unknown columns: {unknowns}
Sample data: {json.dumps(sample_rows[:2], default=str)}
Return ONLY JSON: {{"col": "standard_field"}} or {{"col": "skip"}}"""
    try:
        from anthropic import Anthropic
        result = json.loads(Anthropic(api_key=api_key).messages.create(
            model="claude-sonnet-4-20250514", max_tokens=400,
            messages=[{"role":"user","content":prompt}]
        ).content[0].text.strip())
        for col, mapped in result.items():
            if mapped != "skip" and mapped in STANDARD_SCHEMA:
                mapping[col] = mapped
    except Exception as e:
        print(f"⚠ AI mapping error: {e}")
    return {k: v for k, v in mapping.items() if v}

print("✅ Column mapper ready")

## 9. Data Cleanup Engine
*(unchanged from v1)*

In [ ]:
import re
from datetime import datetime as dt

def clean_column_names(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^\w\s]","",regex=True)
                  .str.replace(r"\s+","_",regex=True))
    return df

def remove_junk_rows(df):
    df = df.dropna(how="all")
    for col in ["category","supplier"]:
        if col in df.columns:
            mask = df[col].astype(str).str.match(r"(?i)^(total|summe|grand total|gesamt)",na=False)
            df = df[~mask]
    return df.drop_duplicates(), {}

def fix_spend_column(series):
    def clean(val):
        if pd.isna(val): return None
        s = str(val).strip()
        neg = s.startswith("(") and s.endswith(")")
        if neg: s = s[1:-1]
        s = re.sub(r"[€$£¥\s]","",s)
        if re.search(r"\d\.\d{3},\d",s): s = s.replace(".","").replace(",",".")
        else: s = s.replace(",","")
        try: return -float(s) if neg else float(s)
        except: return None
    return series.apply(clean)

def full_cleanup(df):
    df = clean_column_names(df)
    df, _ = remove_junk_rows(df)
    for col in [c for c in df.columns if any(k in c for k in ["spend","amount","total","betrag"])]:
        df[col] = fix_spend_column(df[col])
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")
    return df

print("✅ Cleanup engine ready")

## 10. What to Replace in Your Notebook (v1 → v2)

| Old cell | New cell | Action |
|----------|----------|--------|
| Section 2 (Theme) | Section 2 (Theme) | Replace entire cell — new navy/white palette |
| Section 6 (Default Data) | Section 3 (Data Model) | Replace entire cell — new taxonomy, suppliers added |
| Section 7 (Chart Functions) | Section 4 (Chart Functions) | Replace entire cell — fixes + new charts |
| Section 8 (KPI Cards) | Section 5 (KPI Cards) | Replace — minor style update |
| Section 9 (Dashboard) | Section 7 (Dashboard) | Replace entire cell — year selector, drill-down, new layout |
| Sections 3-5 (Mapper/Cleanup/CFO) | Sections 8-9 | Same logic, no changes needed |

### How to do it in VS Code:
1. Open both notebooks side by side (View → Editor Layout → Split Right)
2. Copy each new cell from this notebook
3. Paste over the matching cell in your original notebook
4. Or simply use this notebook as your new main file

### What you do NOT need to change:
- Cell 0 (Setup) — same packages
- Cell 1 (Imports) — same imports
- The CFO export module works with the new data structure automatically
